In [ ]:
!pip install python-resize-image -q
!pip install ultralytics -q


In [ ]:
# Импорт модулей пайплайна
from features import (
    extract_global_color_features_with_mask,
    extract_local_color_features_with_mask,
    extract_shape_features,
    extract_border_features,
    extract_texture_features,
)
from data import build_dataset
from segmentation import main
from threshold_rules import row_to_labels

import json
import pandas as pd
from tqdm import tqdm


## Пайплайн: фичи → классификация из файла → Шаг 1 (метки)

1. **Фичи** генерируются функциями из `features.py`
2. **Классификация** (diagnosis, feature_type, structure, properties) берётся из CSV
3. **Шаг 1**: числа → категориальные метки (пороговые правила)


In [ ]:
# Пути (настрой под свой проект)
META_CSV = "/kaggle/input/classification-results/classification_results.csv"
ROOT_DIR = "/kaggle/input/all-image-skin"
OUTPUT_CSV = "features_dataset.csv"

meta_df = pd.read_csv(META_CSV)
extractors = {
    "global_color": extract_global_color_features_with_mask,
    "local_color": extract_local_color_features_with_mask,
    "shape": extract_shape_features,
    "border": extract_border_features,
    "texture": extract_texture_features,
}

df = build_dataset(
    root_dir=ROOT_DIR,
    meta_table=meta_df,
    mask_fn=main,
    extractors=extractors,
)
df.to_csv(OUTPUT_CSV, index=False)
print(f"Сохранено {len(df)} строк в {OUTPUT_CSV}")


In [ ]:
# Шаг 1: числа → метки
labels_list = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Шаг 1: числа→метки"):
    labels_list.append(row_to_labels(row))

# Все скалярные метки — в отдельные колонки
label_keys = [k for k in labels_list[0] if k != "center_periphery" and isinstance(labels_list[0][k], str)]
for k in label_keys:
    df[f"labels_{k}"] = [l[k] for l in labels_list]
df["labels_step1"] = labels_list

print("Пример меток:")
print(json.dumps(labels_list[0], ensure_ascii=False, indent=2))


In [ ]:
# Распределение меток
label_cols = [c for c in df.columns if c.startswith("labels_") and c != "labels_step1"]
for col in sorted(label_cols):
    print(f"\n{col}:")
    print(df[col].value_counts())


## Этап 2: Модель отбора важных признаков (опционально)

Если обучена модель отбора (см. `train_importance.py`, `IMPORTANCE_README.md`), можно добавить колонку `important_labels` — топ-10 меток по изображению.

In [ ]:
# Раскомментировать после обучения модели и указать путь к чекпоинту:
# from importance_inference import load_model, add_important_labels_to_dataframe
# import torch
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# importance_model = load_model("importance_checkpoints/best.pt", device=device)
# df = add_important_labels_to_dataframe(df, importance_model, device, image_dir=ROOT_DIR, image_col="filename", image_id_col="image_id")
# print("Пример important_labels:", df["important_labels"].iloc[0])